In [2]:
%pip install pandas numpy
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/world_bank_raw.csv")
print(df.shape)
df.head()

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 11.6 MB 1.6 MB/s eta 0:00:01
     |████████████████████████████████| 21.2 MB 6.1 MB/s eta 0:00:011
     |████████████████████████████████| 349 kB 42.0 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 18.3 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
(6377, 9)


,country_code,country_name,year,gdp_growth,inflation,unemployment,govt_debt_pct_gdp,trade_pct_gdp,gdp_per_capita
0,ABW,Aruba,2001,4.182002,2.883604,NaN,NaN,140.391109,29007.738696
1,ABW,Aruba,2002,-0.944953,3.315247,NaN,NaN,133.230771,28535.463948
2,ABW,Aruba,2003,1.110505,3.656365,NaN,NaN,132.794291,28525.807706
3,ABW,Aruba,2004,7.293728,2.529129,NaN,NaN,132.430930,29959.774819
4,ABW,Aruba,2005,-0.383138,3.397787,NaN,NaN,145.050280,29081.706700


In [3]:
print(df.dtypes)
print(df.isnull().sum())

country_code          object
country_name          object
year                   int64
gdp_growth           float64
inflation            float64
unemployment         float64
govt_debt_pct_gdp    float64
trade_pct_gdp        float64
gdp_per_capita       float64
dtype: object
country_code            0
country_name            0
year                    0
gdp_growth            360
inflation            1973
unemployment          847
govt_debt_pct_gdp    5178
trade_pct_gdp        1255
gdp_per_capita        369
dtype: int64


In [5]:
AGGREGATES = [
    "WLD", "EUU", "OED", "LCN", "EAS", "SAS", "SSF",
    "MEA", "ECS", "NAC", "LMY", "HIC", "LIC", "MIC"
]
df = df[~df["country_code"].isin(AGGREGATES)]
df = df[df["year"] >= 2000]
df = df.drop_duplicates(subset=["country_code", "year"])

In [6]:
numeric_cols = ["gdp_growth", "inflation", "unemployment", "govt_debt_pct_gdp", "trade_pct_gdp", "gdp_per_capita"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df[numeric_cols] = df.groupby("country_code")[numeric_cols].transform(lambda x: x.fillna(x.median()))
df = df.dropna(subset=["gdp_growth"])

/Users/catalinacovrig/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/catalinacovrig/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [7]:
def classify_income(gdp_pc):
    if gdp_pc < 1085:
        return "Low Income"
    elif gdp_pc < 4255:
        return "Lower Middle Income"
    elif gdp_pc < 13205:
        return "Upper Middle Income"
    else:
        return "High Income"

df["income_group"] = df["gdp_per_capita"].apply(classify_income)
df["high_inflation"] = (df["inflation"] > 10).astype(int)
df["gdp_growth_lag1"] = df.groupby("country_code")["gdp_growth"].shift(1)
df["gdp_growth_lag2"] = df.groupby("country_code")["gdp_growth"].shift(2)
df = df.dropna(subset=["gdp_growth_lag1", "gdp_growth_lag2"])

In [8]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/world_bank_clean.csv", index=False)
print(f"Clean dataset: {df.shape}")
df.describe()

Clean dataset: (5561, 13)


,year,gdp_growth,inflation,unemployment,govt_debt_pct_gdp,trade_pct_gdp,gdp_per_capita,high_inflation,gdp_growth_lag1,gdp_growth_lag2
count,5561.000000,5561.000000,4449.000000,5005.000000,2085.000000,4951.000000,5556.000000,5561.000000,5561.000000,5561.000000
mean,2013.958641,3.370886,6.523408,7.683516,51.062891,89.520941,14960.192085,0.118324,3.364726,3.371695
std,6.579734,5.592601,16.781441,5.587732,30.743379,58.076684,22628.148030,0.323020,5.640947,5.671514
min,2003.000000,-54.402093,-16.859691,0.100000,-1.170726,1.995412,243.076666,0.000000,-54.402093,-54.402093
25%,2008.000000,1.455875,1.634950,3.984845,29.179291,54.431593,1813.498038,0.000000,1.386448,1.357695
50%,2014.000000,3.571207,3.515576,6.044000,47.062363,75.590404,5571.463506,0.000000,3.558152,3.556105
75%,2020.000000,5.766957,7.126975,9.715000,67.014816,107.746247,18352.860479,0.000000,5.799093,5.818166
max,2025.000000,86.826748,557.201817,37.320000,194.682829,863.195099,247170.219911,1.000000,86.826748,86.826748
